https://github.com/samlhuillier/code-llama-fine-tune-notebook/blob/main/fine-tune-code-llama.ipynb

# A guide to fine-tuning Code Llama

**In this guide I show you how to fine-tune Code Llama to become a beast of an SQL developer. For coding tasks, you can generally get much better performance out of Code Llama than Llama 2, especially when you specialise the model on a particular task:**

- I use the [b-mc2/sql-create-context](https://huggingface.co/datasets/b-mc2/sql-create-context) which is a bunch of text queries and their corresponding SQL queries
- A Lora approach, quantizing the base model to int 8, freezing its weights and only training an adapter
- Much of the code is borrowed from [alpaca-lora](https://github.com/tloen/alpaca-lora), but I refactored it quite a bit for this


### 2. Pip installs


In [1]:
%env CUDA_VISIBLE_DEVICES=2

env: CUDA_VISIBLE_DEVICES=2


In [2]:
#!pip install git+https://github.com/huggingface/transformers.git@main bitsandbytes  # we need latest transformers for this
#!pip install git+https://github.com/huggingface/peft.git@4c611f4
#!pip install datasets==2.10.1
#import locale # colab workaround
#locale.getpreferredencoding = lambda: "UTF-8" # colab workaround
#!pip install wandb
#!pip install scipy

In [7]:
!pip list | grep "transformers\|trl\|peft\|bitsand\|torch"

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


bitsandbytes              0.48.1
peft                      0.17.1
torch                     2.9.0
torchvision               0.24.0
transformers              4.57.1
trl                       0.24.0


I used an A100 GPU machine with Python 3.10 and cuda 11.8 to run this notebook. It took about an hour to run.

### Loading libraries


In [4]:
from datetime import datetime
import os
import sys

import torch
from peft import (
    LoraConfig,
    get_peft_model,
    get_peft_model_state_dict,
    #prepare_model_for_int8_training,
    prepare_model_for_kbit_training,
    set_peft_model_state_dict,
)
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForSeq2Seq
from trl import SFTTrainer, SFTConfig
#from trl import DataCollatorForCompletionOnlyLM
from transformers import DataCollatorForLanguageModeling
#from trl import DataCllatorForCompletionOnlyLM, SFTTrainer, SFTConfig


(If you have import errors, try restarting your Jupyter kernel)


### Load dataset


In [5]:
from datasets import load_dataset
dataset = load_dataset("b-mc2/sql-create-context", split="train")
train_dataset = dataset.train_test_split(test_size=0.1)["train"]
eval_dataset = dataset.train_test_split(test_size=0.1)["test"]

The above pulls the dataset from the Huggingface Hub and splits 10% of it into an evaluation set to check how well the model is doing through training. If you want to load your own dataset do this:

```
train_dataset = load_dataset('json', data_files='train_set.jsonl', split='train')
eval_dataset = load_dataset('json', data_files='validation_set.jsonl', split='train')
```

And if you want to view any samples in the dataset just do something like:``` ```


In [6]:
print(train_dataset[3])

{'answer': 'SELECT MAX(games_played) FROM table_name_85 WHERE position < 4 AND goals_for_against = "29-24"', 'question': 'What is the highest number of games played of the club with a position number less than 4 and a 29-24 goals for/against?', 'context': 'CREATE TABLE table_name_85 (games_played INTEGER, position VARCHAR, goals_for_against VARCHAR)'}


Each entry is made up of a text 'question', the sql table 'context' and the 'answer'.

### Load model
I load code llama from huggingface in int8. Standard for Lora:

In [7]:
!hf auth whoami

user:  daegunlee


In [8]:
base_model = "codellama/CodeLlama-7b-hf"
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-hf")



`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

torch_dtype=torch.float16 means computations are performed using a float16 representation, even though the values themselves are 8 bit ints.

If you get error "ValueError: Tokenizer class CodeLlamaTokenizer does not exist or is not currently imported." Make sure you have transformers version is 4.33.0.dev0 and accelerate is >=0.20.3.


### 3. Check base model
A very good common practice is to check whether a model can already do the task at hand. Fine-tuning is something you want to try to avoid at all cost:


In [9]:
eval_prompt = """
### Human: 
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.

You must output the SQL query that answers the question.
### Input:
Which Class has a Frequency MHz larger than 91.5, and a City of license of hyannis, nebraska?

### Context:
CREATE TABLE table_name_12 (class VARCHAR, frequency_mhz VARCHAR, city_of_license VARCHAR)

### Response:
"""
# {'question': 'Name the comptroller for office of prohibition', 'context': 'CREATE TABLE table_22607062_1 (comptroller VARCHAR, ticket___office VARCHAR)', 'answer': 'SELECT comptroller FROM table_22607062_1 WHERE ticket___office = "Prohibition"'}
model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.



### Human: 
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.

You must output the SQL query that answers the question.
### Input:
Which Class has a Frequency MHz larger than 91.5, and a City of license of hyannis, nebraska?

### Context:
CREATE TABLE table_name_12 (class VARCHAR, frequency_mhz VARCHAR, city_of_license VARCHAR)

### Response:
SELECT * FROM table_name_12 WHERE class > 91.5 AND city_of_license = 'hyannis, nebraska'

### Human: 
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.

You must output the SQL query that answers the question.
### Input:
Which Class has a


I get the output:
```
SELECT * FROM table_name_12 WHERE class > 91.5 AND city_of_license = 'hyannis, nebraska'
```
which is clearly wrong if the input is asking for just class!

### 4. Tokenization
Setup some tokenization settings like left padding because it makes [training use less memory](https://ai.stackexchange.com/questions/41485/while-fine-tuning-a-decoder-only-llm-like-llama-on-chat-dataset-what-kind-of-pa):

In [10]:
#tokenizer.add_eos_token = True
#tokenizer.pad_token_id = 0
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

Setup the tokenize function to make labels and input_ids the same. This is basically what [self-supervised fine-tuning](https://neptune.ai/blog/self-supervised-learning) is:

In [11]:
def tokenize(prompt):
    result = tokenizer(
        prompt,
        truncation=True,
        max_length=512,
        padding=False,
        return_tensors=None,
    )

    # "self-supervised learning" means the labels are also the inputs:
    result["labels"] = result["input_ids"].copy()

    return result

And run convert each data_point into a prompt that I found online that works quite well:

In [12]:
def generate_and_tokenize_prompt(data_point):
    full_prompt =f"""
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.

You must output the SQL query that answers the question.

### Input:
{data_point["question"]}

### Context:
{data_point["context"]}

### Response:
{data_point["answer"]}
"""
    return tokenize(full_prompt)

In [13]:
'''
def generate_prompt(data_point):
    user_prompt =f""" You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
{data_point["question"]}

### Context:
{data_point["context"]}

"""
    assistant = f"""
### Response:
{data_point["answer"]}
"""
    return {'messages': [ 
                          {'role': 'user', 'content': user_prompt},
                          {'role': 'assistant', 'content': assistant}
                        ] }
'''
def generate_prompt(data_point):
    full_prompt =f""" You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
{data_point["question"]}

### Context:
{data_point["context"]}

### Response:
{data_point["answer"]}
"""
    return {'text': full_prompt}


In [14]:
#prompt_train_dataset = train_dataset.map(generate_prompt, remove_columns=train_dataset.column_names)
#prompt_val_dataset = eval_dataset.map(generate_prompt, remove_columns=train_dataset.column_names)

In [15]:
def generate_and_tokenize_prompt(data_point):
    # prompt/completion pair 생성
    prompt = (
        "You are a powerful text-to-SQL model. Your job is to answer questions about a database.\n"
        "You are given a question and context regarding one or more tables.\n"
        "You must output the SQL query that answers the question.\n\n"
        f"### Input:\n{data_point['question']}\n\n"
        f"### Context:\n{data_point['context']}\n\n"
        "### Response:\n"
    ) 

    completion = data_point["answer"] 
    return {"prompt": prompt, "completion": completion}

In [16]:
prompt_train_dataset = train_dataset.map(generate_and_tokenize_prompt,
                                         remove_columns=train_dataset.column_names,
                                         batched=False,
                                         )
prompt_val_dataset = eval_dataset.map(generate_and_tokenize_prompt,
                                      remove_columns=eval_dataset.column_names,
                                      batched=False,
                                     )

Map:   0%|          | 0/70719 [00:00<?, ? examples/s]

Map:   0%|          | 0/7858 [00:00<?, ? examples/s]

In [17]:
#prompt_train_dataset['text'][0]
prompt_train_dataset[0:5]

{'prompt': ['You are a powerful text-to-SQL model. Your job is to answer questions about a database.\nYou are given a question and context regarding one or more tables.\nYou must output the SQL query that answers the question.\n\n### Input:\nName the circuit for round 18\n\n### Context:\nCREATE TABLE table_name_25 (circuit VARCHAR, round VARCHAR)\n\n### Response:',
  'You are a powerful text-to-SQL model. Your job is to answer questions about a database.\nYou are given a question and context regarding one or more tables.\nYou must output the SQL query that answers the question.\n\n### Input:\nFind the total amount of products ordered before 2018-03-17 07:13:53.\n\n### Context:\nCREATE TABLE customer_orders (order_id VARCHAR, order_date INTEGER); CREATE TABLE order_items (order_quantity INTEGER, order_id VARCHAR)\n\n### Response:',
  'You are a powerful text-to-SQL model. Your job is to answer questions about a database.\nYou are given a question and context regarding one or more tables

Reformat to prompt and tokenize each sample:

In [18]:
#tokenized_train_dataset = train_dataset.map(generate_and_tokenize_prompt)
#tokenized_val_dataset = eval_dataset.map(generate_and_tokenize_prompt)

### 5. Setup Lora

In [19]:
model.train() # put model back into training mode
#model = prepare_model_for_int8_training(model)
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=[
    "q_proj",
    "k_proj",
    "v_proj",
    "o_proj",
],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, config)

Optional stuff to setup Weights and Biases to view training graphs:

In [20]:
#wandb_project = "sql-try2-coder"
#if len(wandb_project) > 0:
#    os.environ["WANDB_PROJECT"] = wandb_project


In [21]:
if torch.cuda.device_count() > 1:
    # keeps Trainer from trying its own DataParallelism when more than 1 gpu is available
    model.is_parallelizable = True
    model.model_parallel = True

### 6. Training arguments
If you run out of GPU memory, change per_device_train_batch_size. The gradient_accumulation_steps variable should ensure this doesn't affect batch dynamics during the training run. All the other variables are standard stuff that I wouldn't recommend messing with:

In [22]:
batch_size = 128
per_device_train_batch_size = 16
gradient_accumulation_steps = batch_size // per_device_train_batch_size
output_dir = "sql-code-llama"

training_args = SFTConfig(
        per_device_train_batch_size=per_device_train_batch_size,
        gradient_accumulation_steps=gradient_accumulation_steps,
        warmup_steps=100,
        max_steps=400,
        learning_rate=3e-4,
        fp16=True,
        logging_steps=10,
        optim="adamw_torch",
        eval_strategy="steps", # if val_set_size > 0 else "no",
        save_strategy="steps",
        eval_steps=20,
        save_steps=20,
        output_dir=output_dir,
        # save_total_limit=3,
        load_best_model_at_end=False,
        # ddp_find_unused_parameters=False if ddp else None,
        #group_by_length=True, # group sequences of roughly the same length together to speed up training
        #report_to="wandb", # if use_wandb else "none",
        #run_name=f"codellama-{datetime.now().strftime('%Y-%m-%d-%H-%M')}", # if use_wandb else None,
        #dataset_text_field="text",
        #completion_only_loss=True,  # ✅ prompt 부분은 loss 제외
    )

trainer = SFTTrainer(
    model=model,
    #train_dataset=tokenized_train_dataset,
    #eval_dataset=tokenized_val_dataset,
    train_dataset=prompt_train_dataset,
    eval_dataset=prompt_val_dataset,
    args=training_args,
    #data_collator=DataCollatorForSeq2Seq(
    #    tokenizer, pad_to_multiple_of=8, return_tensors="pt", padding=True
    #),
    processing_class=tokenizer,
    #response_template="### Response:\n",
    #assistant_only_loss=True, #completion_only_loss=True,
    #instruction_template=None,
    #tokenizer=tokenizer,
    #mlm=False,
    #data_collator=DataCollatorForLanguageModeling(
    #    tokenizer=tokenizer,
    #    response_template="### Response:\n",
    #    instruction_template=None,
    #    mlm=False,     
    #    #pad_to_multiple_of=8, return_tensors="pt",
    #    #padding=True,
    #),
)

Adding EOS to train dataset:   0%|          | 0/70719 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/70719 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/70719 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/7858 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/7858 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/7858 [00:00<?, ? examples/s]

In [23]:
prompt_train_dataset[0]

{'prompt': 'You are a powerful text-to-SQL model. Your job is to answer questions about a database.\nYou are given a question and context regarding one or more tables.\nYou must output the SQL query that answers the question.\n\n### Input:\nName the circuit for round 18\n\n### Context:\nCREATE TABLE table_name_25 (circuit VARCHAR, round VARCHAR)\n\n### Response:',
 'completion': ' SELECT circuit FROM table_name_25 WHERE round = 18'}

Then we do some pytorch-related optimisation (which just make training faster but don't affect accuracy):

In [24]:
model.config.use_cache = False

#old_state_dict = model.state_dict
#model.state_dict = (lambda self, *_, **__: get_peft_model_state_dict(self, old_state_dict())).__get__(
#    model, type(model)
#)

#if torch.__version__ >= "2" and sys.platform != "win32":
#    print("compiling the model")
#    model = torch.compile(model)

In [25]:
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: sdfsdfrr. Use `wandb login --relogin` to force relogin


/home/kotech/venv-torch/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
20,0.616900,0.460833,1.962705,348669.000000,0.849986
40,0.166400,0.134718,1.576553,698652.000000,0.964457
60,0.097300,0.085849,1.772396,1045679.000000,0.974217
80,0.063500,0.057850,1.739160,1394907.000000,0.982592


/home/kotech/venv-torch/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/kotech/venv-torch/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/kotech/venv-torch/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/home/kotech/venv-torch/lib/python3.11/site-packages/bitsandbytes/autograd/_functions.py:181: UserWarning: MatMul8bitLt: inputs

KeyboardInterrupt: 

# 여기까지 하고 노트북을 재시작해서 아래부터 수행하면 추론 샘플을 할 수 있어요

### Load the final checkpoint
Now for the moment of truth! Has our work paid off...?

In [1]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer

base_model = "codellama/CodeLlama-7b-hf"
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    load_in_8bit=True,
    torch_dtype=torch.float16,
    device_map="auto",
)
tokenizer = AutoTokenizer.from_pretrained("codellama/CodeLlama-7b-hf")

`torch_dtype` is deprecated! Use `dtype` instead!
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

To load a fine-tuned Lora/Qlora adapter use PeftModel.from_pretrained. ```output_dir``` should be something containing an adapter_config.json and adapter_model.bin:

In [2]:
from peft import PeftModel
#output_dir = "sql-code-llama/checkpoint-400"
output_dir = "sql-code-llama/checkpoint-80"
model = PeftModel.from_pretrained(model, output_dir)

Try the same prompt as before:

In [5]:
eval_prompt = """### Human:
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.

You must output the SQL query that answers the question.
### Input:
Which Class has a Frequency MHz larger than 91.5, and a City of license of hyannis, nebraska?

### Context:
CREATE TABLE table_name_12 (class VARCHAR, frequency_mhz VARCHAR, city_of_license VARCHAR)

### Response: """
eval_prompt = """You are a powerful text-to-SQL model. Your job is to answer questions about a database.
You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
Name the circuit for round 18

### Context:
CREATE TABLE table_name_25 (circuit VARCHAR, round VARCHAR)

### Response:
"""

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=False))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


<s> You are a powerful text-to-SQL model. Your job is to answer questions about a database.
You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
Name the circuit for round 18

### Context:
CREATE TABLE table_name_25 (circuit VARCHAR, round VARCHAR)

### Response:
SELECT circuit FROM table_name_25 WHERE round = "18"</s>


In [4]:
s = prompt_val_dataset['text'][0]
pos = s.find('### Response:\n')
eval_prompt = s[:pos+14]
label = s[pos+14:]
model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))
print()
print(label)

NameError: name 'prompt_val_dataset' is not defined

In [12]:
eval_prompt='### Human:\nYou are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.\nYou must output the SQL query that answers the question.\n\n### Input:\nWhat score did John Mahaffey have?\n\n### Context:\nCREATE TABLE table_name_52 (score VARCHAR, player VARCHAR)\n\n ### Response:'

model_input = tokenizer(eval_prompt, return_tensors="pt").to("cuda")

model.eval()
with torch.no_grad():
    print(tokenizer.decode(model.generate(**model_input, max_new_tokens=100)[0], skip_special_tokens=True))


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


### Human:
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
What score did John Mahaffey have?

### Context:
CREATE TABLE table_name_52 (score VARCHAR, player VARCHAR)

 ### Response:
SELECT score FROM table_name_52 WHERE player = 'John Mahaffey'

### Human:
You are a powerful text-to-SQL model. Your job is to answer questions about a database. You are given a question and context regarding one or more tables.
You must output the SQL query that answers the question.

### Input:
What is the average score for the 2016 season?

### Context:



And the model outputs:
```
SELECT class FROM table_name_12 WHERE frequency_mhz > 91.5 AND city_of_license = "hyannis, nebraska"
```
So it works! If you want to convert your this adapter to a Llama.cpp model to run locally follow my other [guide](https://ragntune.com/blog/A-guide-to-running-Llama-2-qlora-loras-on-Llama.cpp). If you have any questions, shoot me a message on [Elon Musk's website](https://twitter.com/samlhuillier_).


In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
#from datasets import Dataset, load_metric
from datasets import Dataset
from torch.utils.data import DataLoader
import torch

# ⬇️ 2) 초간단 평행 말뭉치 정의 ―———
raw_data = {
    "source": [
        "Hello",                 # 영어 원문
        "I love you",
        "Good morning"
    ],
    "target": [
        "안녕",                   # 한글 번역
        "사랑해",
        "좋은 아침"
    ]
}
dataset = Dataset.from_dict(raw_data)

# ⬇️ 3) 토크나이저 & 모델 선택 ―———
model_name = "t5-small"          # 약 60 MB, 빠른 테스트용
tokenizer = AutoTokenizer.from_pretrained(model_name)
model     = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ⬇️ 4) 전처리 함수 ―———
max_length_src = 32
max_length_tgt = 32

def preprocess(example):
    # ① 입력(encoder) 토큰화
    model_inputs = tokenizer(
        example["source"],
        max_length=max_length_src,
        truncation=True
    )

    # ② 레이블(decoder) 토큰화  — labels 키에 들어가야 함
    with tokenizer.as_target_tokenizer():
        labels = tokenizer(
            example["target"],
            max_length=max_length_tgt,
            truncation=True
        )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_ds = dataset.map(preprocess, remove_columns=dataset.column_names)

# ⬇️ 5) DataCollatorForSeq2Seq 설정 ―———
collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,                 # decoder_input_ids 자동 생성!
    label_pad_token_id=-100      # (기본값) — padding 부분 손실 무시
)

dataloader = DataLoader(
    tokenized_ds,
    batch_size=2,
    shuffle=False,
    collate_fn=collator
)

# ⬇️ 6) 한 배치 꺼내 검사 ―———
batch = next(iter(dataloader))
print("Keys:", batch.keys())
print("input_ids           :", batch["input_ids"].shape)
print("attention_mask      :", batch["attention_mask"].shape)
print("labels              :", batch["labels"].shape)
print("decoder_input_ids   :", batch["decoder_input_ids"].shape)

In [ ]:
batch.keys()

In [ ]:
batch['labels']

In [ ]:
batch['input_ids']

In [ ]:
# ▶︎ 1) 설치가 안 돼 있다면 먼저:
# pip install transformers datasets accelerate trl torch -q

from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import DataCollatorForCompletionOnlyLM
from datasets import Dataset
from torch.utils.data import DataLoader
import torch

# ▶︎ 2) 초간단 챗 데이터 3줄 ----------------------------
dialogues = [
    "### Human: Hello!\n### Assistant: 안녕!",
    "### Human: I love you\n### Assistant: 사랑해!",
    "### Human: Good morning\n### Assistant: 좋은 아침!"
]
dataset = Dataset.from_dict({"text": dialogues})

# ▶︎ 3) 토크나이저·모델 로드  ----------------------------
model_name = "gpt2"                     # 124 M 파라미터, 가장 가벼운 GPT-2
tokenizer  = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token   # GPT-2 는 pad 토큰이 없으므로 EOS로 대체
model      = AutoModelForCausalLM.from_pretrained(model_name)

# ▶︎ 4) 전처리: text → input_ids ----------------------
def preprocess(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        max_length=64            # 테스트라 작게
    )
    return tokens

tokenized_ds = dataset.map(preprocess, remove_columns=["text"])

# ▶︎ 5) DataCollatorForCompletionOnlyLM ---------------
collator = DataCollatorForCompletionOnlyLM(
    tokenizer=tokenizer,
    response_template="### Assistant:",
    instruction_template="### Human:",   # 선택 — 없으면 None
    mlm=False                             # causal LM 이므로 반드시 False
)

loader = DataLoader(
    tokenized_ds,
    batch_size=2,
    collate_fn=collator,
    shuffle=False
)

# ▶︎ 6) 한 배치 꺼내서 구조 확인 ------------------------
batch = next(iter(loader))
print("input_ids shape    :", batch["input_ids"].shape)
print("labels shape       :", batch["labels"].shape)

# labels에서 -100(=무시) 아닌 부분은 응답 토큰
for i in range(batch["labels"].size(0)):
    kept      = (batch["labels"][i] != -100)
    prompt_pt = tokenizer.decode(batch["input_ids"][i][~kept])
    answer_pt = tokenizer.decode(batch["input_ids"][i][kept])
    print(f"\n🟢 Prompt {i+1}:\n{prompt_pt}")
    print(f"🔵 Answer {i+1} (loss 계산 대상):\n{answer_pt}")


In [ ]:
batch["input_ids"]

In [ ]:
batch["labels"]

In [ ]:
print(tokenized_ds['input_ids'][0])
print(tokenized_ds['attention_mask'][0])

In [ ]:
dataset['text']